In [ ]:
# =========================================================
# COMPLETE MULTI-STORE ETL PIPELINE WITH LIVE MONITORING
# =========================================================

import pandas as pd
import glob
import os
import uuid
import time

# Added for the live folder monitoring feature
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

from sqlalchemy import create_engine, text

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# =========================================================
# DATABASE CONNECTION
# =========================================================

username = "root"
password = "1234"
host = "localhost"
database = "storeN"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

# =========================================================
# 1. EXTRACT FUNCTION
# =========================================================

def extract_data(folder_path):
    print("\n==============================")
    print("EXTRACTING FILES")
    print("==============================")

    # Scans for both CSV and Excel files automatically
    files = glob.glob(os.path.join(folder_path, "*.csv")) + glob.glob(os.path.join(folder_path, "*.xlsx"))

    if not files:
        print(f"Warning: No source data files found in: {folder_path}")
        return pd.DataFrame()

    all_dataframes = []

    for file in files:
        print(f"Reading File: {file}")
        if file.endswith('.csv'):
            df = pd.read_csv(file)
        else:
            df = pd.read_excel(file)

        # Track provenance data
        df["Source_File"] = os.path.basename(file)
        all_dataframes.append(df)

    combined_data = pd.concat(all_dataframes, ignore_index=True)
    print("\nFILES MERGED SUCCESSFULLY")
    return combined_data

# =========================================================
# 2. TRANSFORM FUNCTION
# =========================================================

def transform_data(data):
    if data.empty:
        return data

    print("\n==============================")
    print("TRANSFORMING DATA")
    print("==============================")

    # 1. REMOVE DUPLICATES FROM DATAFRAME
    data = data.drop_duplicates()

    # 2. HANDLE MISSING VALUES
    print("Handling Missing Values...")
    data["Qty"] = data["Qty"].fillna(data["Qty"].mean())
    data["Unit_Price"] = data["Unit_Price"].fillna(data["Unit_Price"].mean())
    data = data.dropna(subset=["CustomerID"])

    # 3. DATA TYPE CONVERSION & DATE FIX
    print("Converting Data Types...")
    data["Qty"] = data["Qty"].astype(int)
    data["Unit_Price"] = data["Unit_Price"].astype(float)
    data["CustomerID"] = data["CustomerID"].astype(str)
    data["StoreID"] = data["StoreID"].astype(str)
    
    # FIXED: Replaces manual backslashes '\\' with '/' to avoid DateParseError
    data["SaleDate"] = data["SaleDate"].astype(str).str.replace('\\', '/', regex=False)
    data["SaleDate"] = pd.to_datetime(data["SaleDate"], format="mixed").dt.date

    # 4. TEXT CLEANING
    print("Cleaning Text Fields...")
    text_columns = ["ProductName", "CurrencyType", "CustomerID", "StoreID"]
    for col in text_columns:
        data[col] = data[col].astype(str).str.strip().str.lower()

    # Define a fixed namespace for deterministic UUIDs
    NAMESPACE = uuid.NAMESPACE_DNS

    # 5. GENERATE DETERMINISTIC PRODUCT IDS
    print("Generating Product IDs...")
    data["ProductID"] = data["ProductName"].apply(
        lambda x: str(uuid.uuid5(NAMESPACE, f"prod-{x}"))
    )

    # 6. GENERATE DETERMINISTIC STORE IDS
    print("Generating Store IDs...")
    data["GeneratedStoreID"] = data["StoreID"].apply(
        lambda x: str(uuid.uuid5(NAMESPACE, f"store-{x}"))
    )

    # 7. GENERATE DETERMINISTIC CUSTOMER IDS
    print("Generating Customer IDs...")
    data["GeneratedCustomerID"] = data["CustomerID"].apply(
        lambda x: str(uuid.uuid5(NAMESPACE, f"cust-{x}"))
    )

    # 8. CURRENCY CONVERSION
    print("Converting Currency to OMR...")
    exchange_rates = {"usd": 0.385, "eur": 0.420, "omr": 1}
    data["Unit_Price_OMR"] = data.apply(
        lambda row: row["Unit_Price"] * exchange_rates.get(row["CurrencyType"], 1),
        axis=1
    )

    # 9. CREATE TOTAL PRICE
    print("Creating Total_Price...")
    data["Total_Price"] = data["Qty"] * data["Unit_Price_OMR"]

    # 10. GENERATE DETERMINISTIC SALE IDs (Prevents duplication upon re-saves)
    print("Generating Deterministic Sale IDs...")
    data["SaleID"] = data.apply(
        lambda r: str(uuid.uuid5(NAMESPACE, f"sale-{r['CustomerID']}-{r['SaleDate']}-{r['Qty']}-{r['ProductName']}")),
        axis=1
    )

    print("\nTRANSFORMATION COMPLETED")
    return data

# =========================================================
# 3. CREATE TABLES FUNCTION
# =========================================================

def create_tables():
    print("\n==============================")
    print("CREATING TABLES")
    print("==============================")

    with engine.connect() as conn:
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Product (
            ProductID VARCHAR(255) PRIMARY KEY,
            ProductName VARCHAR(255)
        )
        """))

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Customer (
            CustomerPK VARCHAR(255) PRIMARY KEY,
            OriginalCustomerID VARCHAR(255)
        )
        """))

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Store (
            StorePK VARCHAR(255) PRIMARY KEY,
            OriginalStoreID VARCHAR(255)
        )
        """))

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Sale (
            SaleID VARCHAR(255) PRIMARY KEY,
            ProductID VARCHAR(255),
            CustomerPK VARCHAR(255),
            StorePK VARCHAR(255),
            Qty FLOAT,
            Unit_Price_OMR FLOAT,
            Total_Price FLOAT,
            SaleDate DATE,
            CurrencyType VARCHAR(50),
            Source_File VARCHAR(255),
            FOREIGN KEY (ProductID) REFERENCES Product(ProductID),
            FOREIGN KEY (CustomerPK) REFERENCES Customer(CustomerPK),
            FOREIGN KEY (StorePK) REFERENCES Store(StorePK)
        )
        """))
        conn.commit()
    print("TABLES CREATED SUCCESSFULLY")

# =========================================================
# 4. LOAD FUNCTION (INTELLIGENT UPSERT)
# =========================================================

def load_data(data):
    if data.empty:
        print("No data available to load.")
        return

    print("\n==============================")
    print("LOADING DATA (INTELLIGENT UPSERT)")
    print("==============================")

    product_df = data[["ProductID", "ProductName"]].drop_duplicates()
    
    customer_df = data[["GeneratedCustomerID", "CustomerID"]].drop_duplicates()
    customer_df.columns = ["CustomerPK", "OriginalCustomerID"]

    store_df = data[["GeneratedStoreID", "StoreID"]].drop_duplicates()
    store_df.columns = ["StorePK", "OriginalStoreID"]

    sale_df = data[[
        "SaleID", "ProductID", "GeneratedCustomerID", "GeneratedStoreID",
        "Qty", "Unit_Price_OMR", "Total_Price", "SaleDate", "CurrencyType", "Source_File"
    ]]
    sale_df.columns = [
        "SaleID", "ProductID", "CustomerPK", "StorePK",
        "Qty", "Unit_Price_OMR", "Total_Price", "SaleDate", "CurrencyType", "Source_File"
    ]

    # Staging structure mapping via INSERT IGNORE to drop duplicates automatically
    with engine.begin() as conn:
        for t_name, tracking_df in [("Product", product_df), ("Customer", customer_df), ("Store", store_df)]:
            tracking_df.to_sql(f"stage_{t_name}", con=conn, if_exists="replace", index=False)
            conn.execute(text(f"INSERT IGNORE INTO {t_name} SELECT * FROM stage_{t_name}"))
            conn.execute(text(f"DROP TABLE stage_{t_name}"))

        # Load newly appended transactional entries directly using INSERT IGNORE strategy
        sale_df.to_sql("stage_Sale", con=conn, if_exists="replace", index=False)
        conn.execute(text("INSERT IGNORE INTO Sale SELECT * FROM stage_Sale"))
        conn.execute(text("DROP TABLE stage_Sale"))

    print("DATA LOADED SUCCESSFULLY")

# =========================================================
# 5. VERIFY FUNCTION
# =========================================================

def verify_data():
    print("\n==============================")
    print("VERIFYING DATA FROM DATABASE")
    print("==============================")
    query = "SELECT * FROM Sale LIMIT 5"
    result = pd.read_sql(query, engine)
    print(result)

# =========================================================
# 6. MAIN PIPELINE EXECUTION ENGINE
# =========================================================

def run_etl_pipeline():
    script_dir = os.getcwd()
    data = extract_data(script_dir)
    transformed_data = transform_data(data)
    create_tables()
    load_data(transformed_data)
    verify_data()
    print("\nETL PIPELINE COMPLETED SUCCESSFULLY")

# =========================================================
# 7. LIVE WATCHER EVENT HANDLER (RUNS AUTOMATICALLY ON SAVE)
# =========================================================

class MyFolderHandler(FileSystemEventHandler):
    def __init__(self):
        self.last_run = 0

    def on_modified(self, event):
        if not event.is_directory and event.src_path.endswith(('.csv', '.xlsx', '.xls')):
            current_time = time.time()
            # Anti-bounce safety threshold to block continuous file write triggers
            if current_time - self.last_run > 5:  
                self.last_run = current_time
                print(f"\n[AUTOMATIC DETECT] Manual change saved in file: {os.path.basename(event.src_path)}")
                print("Running ETL pipeline to refresh database...")
                try:
                    run_etl_pipeline()
                except Exception as e:
                    print(f"Pipeline Execution Failed: {e}")

# =========================================================
# RUNNING ENVIRONMENT MAIN STREAM
# =========================================================

if __name__ == "__main__":
    # 1. Run initialization sync cycle first
    run_etl_pipeline()

    # 2. Spin up folder observer live tracking listener thread
    folder_to_watch = os.getcwd()
    print(f"\nMonitoring folder directory: {folder_to_watch}")
    print("Status: Active. Modify your Excel sheet, add Store_C, and save it. Changes will load automatically.")

    event_handler = MyFolderHandler()
    observer = Observer()
    observer.schedule(event_handler, path=folder_to_watch, recursive=False)
    observer.start()

    try:
        while True:
            time.sleep(3) 
    except KeyboardInterrupt:
        print("\nFolder file watcher system stopped.")
        observer.stop()
    observer.join()








EXTRACTING FILES
Reading File: c:\Users\DELL\Desktop\DATA AI\store_sales_1.csv
Reading File: c:\Users\DELL\Desktop\DATA AI\store_sales_2.csv
Reading File: c:\Users\DELL\Desktop\DATA AI\store_sales_3.csv

FILES MERGED SUCCESSFULLY

TRANSFORMING DATA
Handling Missing Values...
Converting Data Types...
Cleaning Text Fields...
Generating Product IDs...
Generating Store IDs...
Generating Customer IDs...
Converting Currency to OMR...
Creating Total_Price...
Generating Deterministic Sale IDs...

TRANSFORMATION COMPLETED

CREATING TABLES
TABLES CREATED SUCCESSFULLY

LOADING DATA (INTELLIGENT UPSERT)


C:\Users\DELL\AppData\Local\Temp\ipykernel_1920\3512290879.py:233: UserWarning: The provided table name 'stage_Product' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  tracking_df.to_sql(f"stage_{t_name}", con=conn, if_exists="replace", index=False)
C:\Users\DELL\AppData\Local\Temp\ipykernel_1920\3512290879.py:233: UserWarning: The provided table name 'stage_Customer' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  tracking_df.to_sql(f"stage_{t_name}", con=conn, if_exists="replace", index=False)
C:\Users\DELL\AppData\Local\Temp\ipykernel_1920\3512290879.py:233: UserWarning: The provided table name 'stage_Store' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  tracking_df.to_sql(f"

DATA LOADED SUCCESSFULLY

VERIFYING DATA FROM DATABASE
                                 SaleID                             ProductID  \
0  013e64b4-a679-584b-9dee-7f773156af96  8294facc-b4ad-5eae-89b6-8ac39226982d   
1  016df6fa-908e-5420-a1d8-edb0416bc89d  47f95f3a-3e47-585b-85b6-c5f8ae451a35   
2  01f49ee8-2e27-534a-a261-00bb5ba527ac  90cf4a97-0b73-5d35-9256-121bd706037e   
3  035c236d-8a0e-5a93-ab86-160d9f2bc89a  3238b101-20a9-5037-8b5f-df9fc8defaea   
4  038215dc-391a-56e7-aa62-5f3bf00d7ee1  7385ec5a-bcac-5a6c-ac69-63c16e11d801   

                             CustomerPK                               StorePK  \
0  e0b1271a-f755-5c45-9298-a21ab62b9ef6  f1347986-c764-5cae-bd4c-dcb7c90b0531   
1  94c0f83a-f992-5e0a-8508-d9d480f8f4e3  f1347986-c764-5cae-bd4c-dcb7c90b0531   
2  8d318212-eaa1-5961-94e8-3be114d128a8  f1347986-c764-5cae-bd4c-dcb7c90b0531   
3  464ca898-61fa-5a6d-b38f-0a5952ff72a9  2393a29d-b40e-5662-a2bd-91ef8537701c   
4  47a0dc66-ec07-55db-be7c-5d8f18167de3  2393a29d-b40

C:\Users\DELL\AppData\Local\Temp\ipykernel_1920\3512290879.py:233: UserWarning: The provided table name 'stage_Product' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  tracking_df.to_sql(f"stage_{t_name}", con=conn, if_exists="replace", index=False)
C:\Users\DELL\AppData\Local\Temp\ipykernel_1920\3512290879.py:233: UserWarning: The provided table name 'stage_Customer' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  tracking_df.to_sql(f"stage_{t_name}", con=conn, if_exists="replace", index=False)
C:\Users\DELL\AppData\Local\Temp\ipykernel_1920\3512290879.py:233: UserWarning: The provided table name 'stage_Store' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  tracking_df.to_sql(f"

DATA LOADED SUCCESSFULLY

VERIFYING DATA FROM DATABASE
                                 SaleID                             ProductID  \
0  013e64b4-a679-584b-9dee-7f773156af96  8294facc-b4ad-5eae-89b6-8ac39226982d   
1  016df6fa-908e-5420-a1d8-edb0416bc89d  47f95f3a-3e47-585b-85b6-c5f8ae451a35   
2  01f49ee8-2e27-534a-a261-00bb5ba527ac  90cf4a97-0b73-5d35-9256-121bd706037e   
3  035c236d-8a0e-5a93-ab86-160d9f2bc89a  3238b101-20a9-5037-8b5f-df9fc8defaea   
4  038215dc-391a-56e7-aa62-5f3bf00d7ee1  7385ec5a-bcac-5a6c-ac69-63c16e11d801   

                             CustomerPK                               StorePK  \
0  e0b1271a-f755-5c45-9298-a21ab62b9ef6  f1347986-c764-5cae-bd4c-dcb7c90b0531   
1  94c0f83a-f992-5e0a-8508-d9d480f8f4e3  f1347986-c764-5cae-bd4c-dcb7c90b0531   
2  8d318212-eaa1-5961-94e8-3be114d128a8  f1347986-c764-5cae-bd4c-dcb7c90b0531   
3  464ca898-61fa-5a6d-b38f-0a5952ff72a9  2393a29d-b40e-5662-a2bd-91ef8537701c   
4  47a0dc66-ec07-55db-be7c-5d8f18167de3  2393a29d-b40

In [ ]:
# =========================================================
# DEDICATED LIVE WEATHER API ETL PIPELINE
# =========================================================

import pandas as pd
import uuid
import time
import requests
import schedule
from datetime import datetime
from sqlalchemy import create_engine, text

# =========================================================
# CONFIGURATION & CONNECTIONS
# =========================================================

username = "root"
password = "1234"
host = "localhost"
database = "weatherDB"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

# ⚠️ REPLACE WITH YOUR ACTUAL OPENWEATHER API KEY AND TARGET CITIES
WEATHER_API_KEY = "4ff7abb2cd202badcf303f6264c2325e"
CITIES = ["muscat", "seeb", "dubai"] 

# =========================================================
# 1. EXTRACT FUNCTION (LIVE OPENWEATHER API)
# =========================================================

def extract_weather():
    print("\n==============================")
    print("EXTRACTING LIVE WEATHER DATA")
    print("==============================")
    
    raw_records = []
    for city in CITIES:
        url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units=metric"
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                json_data = response.json()
                raw_records.append(json_data)
                print(f"Successfully fetched API data for: {city}")
            else:
                print(f"Failed to fetch {city}: Status {response.status_code}")
        except Exception as e:
            print(f"Connection error for {city}: {e}")
            
    return raw_records

# =========================================================
# 2. TRANSFORM FUNCTION
# =========================================================

def transform_weather(raw_data):
    if not raw_data:
        return pd.DataFrame()

    print("\n==============================")
    print("TRANSFORMING WEATHER METRICS")
    print("==============================")
    
    transformed_records = []
    NAMESPACE = uuid.NAMESPACE_DNS
    
    for item in raw_data:
        city_name = str(item.get("name")).strip().lower()
        country_code = str(item.get("sys", {}).get("country")).strip().lower()
        condition_desc = str(item["weather"][0]["description"]).strip().lower()
        
        # 1. Generate clean, predictable surrogate keys using UUIDv5
        location_id = str(uuid.uuid5(NAMESPACE, f"loc-{city_name}"))
        condition_id = str(uuid.uuid5(NAMESPACE, f"cond-{condition_desc}"))
        
        # 2. Round values and extract exact timestamps
        temp = round(float(item["main"]["temp"]), 2)
        humidity = int(item["main"]["humidity"])
        wind_speed = round(float(item["wind"]["speed"]), 2)
        
        # OpenWeather returns a Unix timestamp (dt); convert it to a standard database datetime string
        recorded_at = datetime.fromtimestamp(item["dt"]).strftime('%Y-%m-%d %H:%M:%S')
        
        # 3. Create a unique, deterministic primary key for this exact reading instance
        reading_id = str(uuid.uuid5(NAMESPACE, f"read-{location_id}-{recorded_at}"))
        
        transformed_records.append({
            "ReadingID": reading_id,
            "LocationID": location_id,
            "CityName": city_name,
            "Country": country_code,
            "ConditionID": condition_id,
            "Description": condition_desc,
            "Temperature": temp,
            "Humidity": humidity,
            "WindSpeed": wind_speed,
            "RecordedAt": recorded_at
        })
        
    return pd.DataFrame(transformed_records)

# =========================================================
# 3. CREATE TABLES FUNCTION (DEDICATED WEATHER SCHEMA)
# =========================================================

def create_weather_tables():
    print("\n==============================")
    print("INITIALIZING WEATHER TABLES")
    print("==============================")
    with engine.connect() as conn:
        # Location Dimension
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Location (
            LocationID VARCHAR(255) PRIMARY KEY,
            CityName VARCHAR(255),
            Country VARCHAR(50)
        )
        """))
        
        # Condition Dimension
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS WeatherCondition (
            ConditionID VARCHAR(255) PRIMARY KEY,
            Description VARCHAR(255)
        )
        """))
        
        # Weather Readings Fact Table
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS WeatherReading (
            ReadingID VARCHAR(255) PRIMARY KEY,
            LocationID VARCHAR(255),
            ConditionID VARCHAR(255),
            Temperature FLOAT,
            Humidity INT,
            WindSpeed FLOAT,
            RecordedAt DATETIME,
            FOREIGN KEY (LocationID) REFERENCES Location(LocationID),
            FOREIGN KEY (ConditionID) REFERENCES WeatherCondition(ConditionID)
        )
        """))
        conn.commit()
    print("DATABASE STRUCTURE READY")

# =========================================================
# 4. LOAD FUNCTION (INTELLIGENT UPSERT)
# =========================================================

def load_weather(df):
    if df.empty:
        print("No processed data available to write.")
        return

    print("\n==============================")
    print("LOADING WITH INTELLIGENT UPSERT")
    print("==============================")
    
    # Split main dataframe back into distinct normalized entity subsets
    location_df = df[["LocationID", "CityName", "Country"]].drop_duplicates()
    condition_df = df[["ConditionID", "Description"]].drop_duplicates()
    reading_df = df[["ReadingID", "LocationID", "ConditionID", "Temperature", "Humidity", "WindSpeed", "RecordedAt"]].drop_duplicates()
    
    with engine.begin() as conn:
        # Load Dimensions smoothly using temporary staging tables + INSERT IGNORE
        for table_name, sub_df in [("Location", location_df), ("WeatherCondition", condition_df)]:
            sub_df.to_sql(f"stage_{table_name}", con=conn, if_exists="replace", index=False)
            conn.execute(text(f"INSERT IGNORE INTO {table_name} SELECT * FROM stage_{table_name}"))
            conn.execute(text(f"DROP TABLE stage_{table_name}"))
            
        # Load Facts metrics
        reading_df.to_sql("stage_WeatherReading", con=conn, if_exists="replace", index=False)
        conn.execute(text("INSERT IGNORE INTO WeatherReading SELECT * FROM stage_WeatherReading"))
        conn.execute(text("DROP TABLE stage_WeatherReading"))
        
    print("DATABASE LOAD COMPLETE")

# =========================================================
# 5. CORE PIPELINE CONTROLLER
# =========================================================

def run_weather_etl():
    try:
        raw_payloads = extract_weather()
        processed_df = transform_weather(raw_payloads)
        load_weather(processed_df)
        print(f"\n[SUCCESS] Weather stream logged successfully at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    except Exception as e:
        print(f"\n[CRITICAL FAILURE] Pipeline halted: {e}")

# =========================================================
# AUTOMATED TICKER LOOP
# =========================================================

if __name__ == "__main__":
    # 1. Build infrastructure
    create_weather_tables()
    
    # 2. Immediate baseline run on boot
    run_weather_etl()
    
    # 3. Schedule recurring intervals precisely every 5 minutes
    schedule.every(5).minutes.do(run_weather_etl)
    
    print("\n=============================================")
    print("[ETL ENGINE ONLINE] Polling OpenWeather every 5 minutes.")
    print("Press Ctrl+C to terminate execution cleanly.")
    print("=============================================")
    
    try:
        while True:
            schedule.run_pending()
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nScheduler stopped cleanly. Goodbye!")


INITIALIZING WEATHER TABLES
DATABASE STRUCTURE READY

EXTRACTING LIVE WEATHER DATA
Successfully fetched API data for: muscat
Successfully fetched API data for: seeb
Successfully fetched API data for: dubai

TRANSFORMING WEATHER METRICS

LOADING WITH INTELLIGENT UPSERT
DATABASE LOAD COMPLETE

[SUCCESS] Weather stream logged successfully at 2026-06-08 12:45:23

[ETL ENGINE ONLINE] Polling OpenWeather every 5 minutes.
Press Ctrl+C to terminate execution cleanly.


C:\Users\DELL\AppData\Local\Temp\ipykernel_18188\3951908889.py:167: UserWarning: The provided table name 'stage_Location' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  sub_df.to_sql(f"stage_{table_name}", con=conn, if_exists="replace", index=False)
C:\Users\DELL\AppData\Local\Temp\ipykernel_18188\3951908889.py:167: UserWarning: The provided table name 'stage_WeatherCondition' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  sub_df.to_sql(f"stage_{table_name}", con=conn, if_exists="replace", index=False)
C:\Users\DELL\AppData\Local\Temp\ipykernel_18188\3951908889.py:172: UserWarning: The provided table name 'stage_WeatherReading' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  re